# Laboratório — Auto-informação e entropia

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/21-entropia-auto-informacao-laboratorio.ipynb)

Este laboratório calcula e visualiza informação de Shannon com dados discretos. Tudo é gerado localmente.

**Dependências:** Python 3.11+, NumPy 2.0+, pandas 2.0+ e Matplotlib 3.8+.

**Reprodutibilidade:** seed global `20260907`; versões são exibidas; a última célula valida numericamente os resultados.

## Roteiro

1. implementar funções seguras;
2. converter bits, nats e hartleys;
3. visualizar a entropia Bernoulli;
4. decompor uma distribuição categórica;
5. verificar o máximo na uniforme;
6. simular o viés do estimador *plug-in*;
7. construir um código de Huffman;
8. calcular ganho de informação;
9. alterar a entropia de tokens com temperatura.

In [ ]:
import heapq
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
rng = np.random.default_rng(SEED)

print(f"Python {sys.version.split()[0]}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Funções numericamente seguras

Na entropia, termos com probabilidade zero contribuem com zero pelo limite. Na auto-informação, probabilidade zero produz infinito. A base deve ser positiva e diferente de 1.

In [ ]:
def _validar_base(base):
    if base <= 0 or np.isclose(base, 1):
        raise ValueError("A base deve ser positiva e diferente de 1.")


def auto_informacao(p, base=2):
    _validar_base(base)
    arr = np.asarray(p, dtype=float)
    if np.any((arr < 0) | (arr > 1)):
        raise ValueError("Probabilidades devem estar em [0, 1].")
    saida = np.full(arr.shape, np.inf, dtype=float)
    mask = arr > 0
    saida[mask] = -np.log(arr[mask]) / np.log(base)
    return float(saida) if saida.ndim == 0 else saida


def entropia(p, base=2):
    _validar_base(base)
    arr = np.asarray(p, dtype=float)
    if arr.ndim != 1 or np.any(arr < 0) or not np.isclose(arr.sum(), 1.0):
        raise ValueError("p deve ser um vetor não negativo que soma 1.")
    mask = arr > 0
    return float(-np.sum(arr[mask] * np.log(arr[mask])) / np.log(base))


print("I(1) =", auto_informacao(1.0), "bit")
print("I(0,5) =", auto_informacao(0.5), "bit")
print("I(0) =", auto_informacao(0.0), "bit")
print("H([0,7; 0,3; 0]) =", entropia([0.7, 0.3, 0.0]), "bits")

## 2. A mesma surpresa em unidades diferentes

O evento (p=1/8) exige três decisões binárias ideais. A mudança de base altera apenas a unidade.

In [ ]:
probabilidades = np.array([1, 1/2, 1/4, 1/8, 1/100])
unidades = pd.DataFrame({
    "p": probabilidades,
    "bits": auto_informacao(probabilidades, 2),
    "nats": auto_informacao(probabilidades, np.e),
    "hartleys": auto_informacao(probabilidades, 10),
})
print(unidades.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

conversao = unidades["bits"] * np.log(2)
print("\nMaior erro em bits × ln(2) = nats:", np.max(np.abs(conversao-unidades["nats"])))

## 3. Entropia de Bernoulli

A incerteza é zero nos extremos determinísticos e máxima em (p=0,5). Calculamos os extremos separadamente para não avaliar `log(0)`.

In [ ]:
p_grid = np.linspace(0, 1, 1001)
h_grid = np.array([entropia([p, 1-p]) for p in p_grid])
indice_max = np.argmax(h_grid)

print(f"Máximo numérico: p={p_grid[indice_max]:.3f}; H={h_grid[indice_max]:.6f} bit")
print(f"H(p=0,9)={entropia([0.9,0.1]):.6f} bit")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p_grid, h_grid, color="#6f42c1", linewidth=2)
ax.scatter([0.5], [1.0], color="#d62728", zorder=3, label="máximo: moeda justa")
ax.set(xlabel="p = P(X=1)", ylabel="H₂(p) em bits",
       title="Entropia da distribuição Bernoulli")
ax.set_ylim(-0.03, 1.05)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Decomposição de uma distribuição categórica

Um resultado raro tem muita auto-informação, mas pequeno peso na esperança. A soma da última coluna é a entropia.

In [ ]:
simbolos = np.array(["A", "B", "C", "D"])
p_cat = np.array([0.5, 0.25, 0.125, 0.125])
i_cat = auto_informacao(p_cat)
contrib = p_cat * i_cat
tabela = pd.DataFrame({
    "símbolo": simbolos,
    "p(x)": p_cat,
    "I(x) bits": i_cat,
    "p(x)I(x)": contrib,
})
print(tabela.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\nH(X)={contrib.sum():.6f} bits")

amostra = rng.choice(simbolos, size=100_000, p=p_cat)
mapa_i = dict(zip(simbolos, i_cat))
media_surpresa = np.mean([mapa_i[x] for x in amostra])
print(f"Média simulada da surpresa={media_surpresa:.6f} bits")

## 5. Máximo no uniforme

Para (K=8), o limite é (log_2 8=3) bits. Sorteamos milhares de distribuições no simplex; nenhuma deve superar a uniforme.

In [ ]:
k = 8
uniforme = np.full(k, 1/k)
h_uniforme = entropia(uniforme)
distribuicoes = rng.dirichlet(np.ones(k), size=20_000)
h_aleatorias = np.array([entropia(p) for p in distribuicoes])

print(f"H(uniforme)={h_uniforme:.6f} bits")
print(f"Maior H entre 20.000 sorteios={h_aleatorias.max():.6f} bits")
print(f"Menor H entre 20.000 sorteios={h_aleatorias.min():.6f} bits")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(h_aleatorias, bins=50, color="#4c78a8", alpha=0.85)
ax.axvline(h_uniforme, color="#d62728", linestyle="--", linewidth=2,
           label=f"máximo teórico = {h_uniforme:.1f} bits")
ax.set(xlabel="Entropia (bits)", ylabel="Número de distribuições",
       title="Distribuições categóricas aleatórias com oito classes")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Estimação e viés de amostra pequena

A fonte verdadeira tem 20 categorias desbalanceadas. Em cada tamanho amostral, repetimos a coleta 2.000 vezes. Miller–Madow adiciona ((K_{obs}-1)/(2n\ln2)) ao estimador em bits.

In [ ]:
pesos = 1 / np.arange(1, 21, dtype=float) ** 1.2
p_true = pesos / pesos.sum()
h_true = entropia(p_true)

def entropia_contagens(contagens):
    contagens = np.asarray(contagens)
    p = contagens[contagens > 0] / contagens.sum()
    return entropia(p)


def miller_madow_bits(contagens):
    h_mle = entropia_contagens(contagens)
    k_obs = np.count_nonzero(contagens)
    n = np.sum(contagens)
    return h_mle + (k_obs - 1) / (2 * n * np.log(2))


linhas = []
for n in [20, 50, 100, 500, 2_000]:
    contagens = rng.multinomial(n, p_true, size=2_000)
    h_mle = np.array([entropia_contagens(c) for c in contagens])
    h_mm = np.array([miller_madow_bits(c) for c in contagens])
    linhas.append({
        "n": n,
        "H verdadeiro": h_true,
        "média plug-in": h_mle.mean(),
        "viés plug-in": h_mle.mean()-h_true,
        "média Miller-Madow": h_mm.mean(),
        "viés Miller-Madow": h_mm.mean()-h_true,
    })
estimacao = pd.DataFrame(linhas)
print(estimacao.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

## 7. Código de Huffman

O algoritmo combina repetidamente os dois símbolos menos prováveis. A implementação retorna comprimentos, suficientes para calcular o comprimento médio e verificar (H\le L<H+1).

In [ ]:
def comprimentos_huffman(probabilidades):
    p = np.asarray(probabilidades, dtype=float)
    if np.any(p <= 0) or not np.isclose(p.sum(), 1):
        raise ValueError("Huffman requer probabilidades positivas que somam 1.")
    if len(p) == 1:
        return np.array([0])
    contador = itertools.count()
    heap = [(prob, next(contador), [i]) for i, prob in enumerate(p)]
    heapq.heapify(heap)
    comprimentos = np.zeros(len(p), dtype=int)
    while len(heap) > 1:
        p1, _, s1 = heapq.heappop(heap)
        p2, _, s2 = heapq.heappop(heap)
        comprimentos[s1] += 1
        comprimentos[s2] += 1
        heapq.heappush(heap, (p1+p2, next(contador), s1+s2))
    return comprimentos


p_codigo = np.array([0.40, 0.30, 0.20, 0.10])
lengths = comprimentos_huffman(p_codigo)
h_codigo = entropia(p_codigo)
l_medio = np.dot(p_codigo, lengths)
kraft = np.sum(2.0 ** (-lengths))
print(pd.DataFrame({"p": p_codigo, "comprimento": lengths}).to_string(index=False))
print(f"H={h_codigo:.6f} bits; L={l_medio:.6f} bits; soma de Kraft={kraft:.6f}")

## 8. Ganho de informação em árvore

O nó pai contém seis positivos e quatro negativos. O corte produz um filho puro com quatro positivos e outro com dois positivos e quatro negativos.

In [ ]:
def entropia_contagem_binaria(positivos, negativos):
    total = positivos + negativos
    return entropia([positivos/total, negativos/total])


h_pai = entropia_contagem_binaria(6, 4)
h_esquerda = entropia_contagem_binaria(4, 0)
h_direita = entropia_contagem_binaria(2, 4)
h_filhos = (4/10)*h_esquerda + (6/10)*h_direita
ganho = h_pai - h_filhos

print(f"H(pai)={h_pai:.6f} bits")
print(f"H ponderada dos filhos={h_filhos:.6f} bits")
print(f"Ganho de informação={ganho:.6f} bit")

## 9. Temperatura e entropia de próximo token

Usamos *softmax* estável. Aumentar a temperatura achata esta distribuição fixa; não transforma probabilidades em verdade nem melhora automaticamente a geração.

In [ ]:
def softmax_temperatura(logits, temperatura):
    if temperatura <= 0:
        raise ValueError("Temperatura deve ser positiva.")
    z = np.asarray(logits, dtype=float) / temperatura
    exp_z = np.exp(z-z.max())
    return exp_z / exp_z.sum()


tokens = ["modelo", "sistema", "agente", "dado"]
logits = np.array([3.0, 1.0, 0.0, -1.0])
temperaturas = [0.25, 0.5, 1.0, 2.0, 5.0]
linhas_temp = []
for t in temperaturas:
    p = softmax_temperatura(logits, t)
    linhas_temp.append({"T": t, "H(bits)": entropia(p), **dict(zip(tokens, p))})
temp_df = pd.DataFrame(linhas_temp)
print(temp_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(temp_df["T"], temp_df["H(bits)"], marker="o", color="#2ca02c")
ax.axhline(np.log2(len(tokens)), color="#333333", linestyle="--", label="uniforme: 2 bits")
ax.set(xlabel="Temperatura", ylabel="Entropia (bits)",
       title="Temperatura maior espalha a distribuição deste exemplo")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 10. Verificações automáticas

As asserções cobrem identidades exatas e fenômenos simulados. Tolerâncias explícitas tornam falhas visíveis sem exigir igualdade bit a bit em toda versão de dependência.

In [ ]:
assert auto_informacao(1.0) == 0.0
assert np.isinf(auto_informacao(0.0))
assert np.isclose(auto_informacao(1/8), 3.0)
assert np.isclose(entropia([0.5,0.5]), 1.0)
assert np.isclose(entropia([1.0,0.0]), 0.0)
assert np.isclose(entropia(p_cat), 1.75)
assert abs(media_surpresa-entropia(p_cat)) < 0.02
assert np.all(h_aleatorias <= h_uniforme + 1e-12)
assert estimacao.loc[estimacao["n"] == 20, "viés plug-in"].iloc[0] < -0.25
assert abs(estimacao.iloc[-1]["viés plug-in"]) < 0.02
assert h_codigo <= l_medio < h_codigo + 1
assert np.isclose(kraft, 1.0)
assert np.isclose(ganho, 0.419973094, atol=1e-8)
assert np.all(np.diff(temp_df["H(bits)"]) > 0)

print("Todas as verificações foram aprovadas.")
print(f"Bernoulli máxima: H={h_grid[indice_max]:.6f} bit em p={p_grid[indice_max]:.3f}")
print(f"Categórica: H={entropia(p_cat):.6f}; surpresa simulada={media_surpresa:.6f} bits")
print(f"Viés plug-in: n=20 -> {estimacao.iloc[0]['viés plug-in']:.6f}; n=2000 -> {estimacao.iloc[-1]['viés plug-in']:.6f}")
print(f"Huffman: H={h_codigo:.6f}; L={l_medio:.6f} bits")
print(f"Ganho da árvore={ganho:.6f} bit")
print(f"Temperatura: H(T=0,25)={temp_df.iloc[0]['H(bits)']:.6f}; H(T=5)={temp_df.iloc[-1]['H(bits)']:.6f}")

## Conclusões

- O logaritmo converte a multiplicação de probabilidades independentes em soma de informação.
- A entropia é a média ponderada da surpresa, não a média simples dos valores de auto-informação.
- A distribuição uniforme maximiza entropia em suporte finito.
- Entropia limita o comprimento médio de códigos prefixos.
- Frequências empíricas pequenas subestimam a incerteza quando categorias raras não aparecem.
- Ganho de informação mede redução de impureza em uma divisão preditiva.
- Temperatura altera a concentração de uma distribuição, mas não garante correção ou calibração.

Volte à [Aula 21](../aulas/21-entropia-auto-informacao.md) para exemplos resolvidos, checklist e referências.